# ASL: Train a CNN Baseline on Kaggle

Select your **Kaggle remote Python kernel** in VS Code and run the cells in order. This notebook is self-contained and does not import local project files.

The default is a **1,000-sample pipeline check with 2 epochs**, not a meaningful 250-class benchmark. After it succeeds, set `SAMPLE_LIMIT = None` and `EPOCHS = 20`, then rerun from the configuration cell. A GPU is optional for the small check and useful for full training; verify Kaggle's accelerator settings before starting a full run.

Each run saves its own directory under `/kaggle/working/asl-project`. Download important results before ending the Kaggle session. No training starts until you execute the training cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
SAMPLE_LIMIT = 1000  # Set to None to use the full dataset.
EPOCHS = 2          # Use 20 for an initial full-data experiment.
N_FRAMES = 32
BATCH_SIZE = 64
VALIDATION_FRACTION = 0.2
DATA_DIR = None     # Or Path('/kaggle/input/competitions/asl-signs').
INPUT_DIR = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/asl-project')

if not INPUT_DIR.is_dir():
    raise RuntimeError('Select a Kaggle remote kernel; /kaggle/input is unavailable.')
if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < 2:
    raise ValueError('SAMPLE_LIMIT must be None or at least 2.')
tf.keras.utils.set_random_seed(SEED)
print('Python:', sys.executable)
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))
print('Mode:', 'FULL DATASET' if SAMPLE_LIMIT is None else 'PIPELINE CHECK')


In [ ]:
# To specify the dataset manually, use Path('/kaggle/input/competitions/asl-signs').
# Search only a few directory levels to avoid traversing every parquet file.
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            'ASL competition data was not found. In your Kaggle notebook, use Add Input to add '
            'Google - Isolated Sign Language Recognition (asl-signs), then rerun this cell.'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple dataset directories found; set DATA_DIR explicitly above: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv is missing required columns: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv contains no samples')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('Label indices must be contiguous and start at zero')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv contains labels missing from the official label mapping')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())


## Split by signer before sampling

The split uses all metadata first. A signer belongs to only one partition. In pipeline-check mode, samples are then selected independently from each partition. Many classes may be absent in a small run, so its accuracy should not be reported as a full-task result.


In [ ]:
if table['participant_id'].nunique() < 2:
    raise ValueError('At least two distinct signers are required.')
splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED)
train_indices, val_indices = next(splitter.split(table, groups=table['participant_id']))
train_rows = table.iloc[train_indices].copy()
val_rows = table.iloc[val_indices].copy()
train_signers = set(train_rows['participant_id'])
val_signers = set(val_rows['participant_id'])
assert train_signers.isdisjoint(val_signers), 'Signer leakage detected.'

if SAMPLE_LIMIT is not None:
    train_count = max(1, int(SAMPLE_LIMIT * (1 - VALIDATION_FRACTION)))
    val_count = max(1, SAMPLE_LIMIT - train_count)
    train_rows = train_rows.sample(n=min(train_count, len(train_rows)), random_state=SEED)
    val_rows = val_rows.sample(n=min(val_count, len(val_rows)), random_state=SEED)
train_rows = train_rows.reset_index(drop=True)
val_rows = val_rows.reset_index(drop=True)
N_CLASSES = len(labels)
print('Training samples:', len(train_rows), '| classes:', train_rows['sign'].nunique())
print('Validation samples:', len(val_rows), '| classes:', val_rows['sign'].nunique())
print('Training signers:', sorted(train_signers))
print('Validation signers:', sorted(val_signers))
print('Validation classes absent from selected training data:',
      len(set(val_rows['sign']) - set(train_rows['sign'])))

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = OUTPUT_ROOT / ('cnn_' + run_id)
RUN_DIR.mkdir(parents=True, exist_ok=False)
config = dict(seed=SEED, sample_limit=SAMPLE_LIMIT, epochs=EPOCHS,
              frames=N_FRAMES, batch_size=BATCH_SIZE,
              validation_fraction=VALIDATION_FRACTION, data_dir=str(DATA_DIR),
              tensorflow_version=tf.__version__, n_classes=N_CLASSES,
              train_samples=len(train_rows), validation_samples=len(val_rows),
              train_signers=[str(x) for x in sorted(train_signers)],
              validation_signers=[str(x) for x in sorted(val_signers)],
              feature_version='hands_xy_mask_frame_center_scale_v1')
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(RUN_DIR / 'labels.json').write_text(json.dumps(labels, indent=2), encoding='utf-8')
train_rows.to_csv(RUN_DIR / 'train_split.csv', index=False)
val_rows.to_csv(RUN_DIR / 'validation_split.csv', index=False)
print('Run directory:', RUN_DIR)


## Preprocess raw landmark sequences

Use both hands' xy coordinates and retain a validity mask. Drop frames with no visible hand, center and scale the visible points within each frame, then uniformly resample to 32 frames. Entirely missing-hand sequences become zeros. This is a simple baseline that discards some position and timing information.

Features are written to disk-backed NumPy arrays. Training reads one batch at a time rather than duplicating the full dataset in RAM. Full preprocessing still requires disk space and may take a while.


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def preprocess(frames, length):
    """Official row order: face, left hand, pose, right hand.

    Use xy from both hands. Retain a validity mask; normalize each frame
    relative to the visible hand centroid with a shared xy scale.
    """
    import numpy as np
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    hands, valid = hands[useful], valid[useful]
    if not len(hands):
        return np.zeros((length, 126), dtype=np.float32)
    safe = np.where(valid[..., None], hands, 0)
    count = np.maximum(valid.sum(axis=1), 1)[:, None, None]
    center = safe.sum(axis=1, keepdims=True) / count
    centered = np.where(valid[..., None], safe - center, 0)
    scale = np.sqrt((centered ** 2).sum(axis=(1, 2), keepdims=True) / (count * 2))
    normalized = centered / np.maximum(scale, 1e-6)
    features = np.concatenate([normalized.reshape(len(hands), 84), valid], axis=1)
    # Uniform resampling also repeats frames for short clips.
    indices = np.rint(np.linspace(0, len(features) - 1, length)).astype(int)
    return features[indices].astype(np.float32)


In [ ]:
def prepare_partition(rows, name):
    feature_path = RUN_DIR / f'{name}_features.npy'
    x = np.lib.format.open_memmap(
        feature_path, mode='w+', dtype=np.float32,
        shape=(len(rows), N_FRAMES, 126))
    y = rows['sign'].map(labels).to_numpy(dtype=np.int64)
    empty_count = 0
    for index, row in enumerate(rows.itertuples(index=False)):
        features = preprocess(load_frames(DATA_DIR / row.path), N_FRAMES)
        if not np.isfinite(features).all():
            raise ValueError(f'Non-finite features for {row.path}')
        x[index] = features
        empty_count += int(not features[:, 84:].any())
        if (index + 1) % 500 == 0 or index + 1 == len(rows):
            print(f'{name}: {index + 1}/{len(rows)}', flush=True)
    x.flush()
    del x
    np.save(RUN_DIR / f'{name}_labels.npy', y)
    print(f'{name}: {empty_count} sequences have no valid hand points.')
    return np.load(feature_path, mmap_mode='r'), y

estimated_bytes = (len(train_rows) + len(val_rows)) * N_FRAMES * 126 * 4
print(f'Feature cache estimate: {estimated_bytes / 1024**3:.2f} GiB')
x_train, y_train = prepare_partition(train_rows, 'train')
x_val, y_val = prepare_partition(val_rows, 'validation')
print('Train:', x_train.shape, '| validation:', x_val.shape)


In [ ]:
def make_dataset(x, y, shuffle):
    rng = np.random.default_rng(SEED)
    def batches():
        indices = rng.permutation(len(y)) if shuffle else np.arange(len(y))
        for start in range(0, len(y), BATCH_SIZE):
            batch_indices = indices[start:start + BATCH_SIZE]
            yield np.asarray(x[batch_indices], dtype=np.float32), y[batch_indices]
    dataset = tf.data.Dataset.from_generator(
        batches,
        output_signature=(
            tf.TensorSpec((None, N_FRAMES, 126), tf.float32),
            tf.TensorSpec((None,), tf.int64)))
    return dataset.apply(tf.data.experimental.assert_cardinality(
        math.ceil(len(y) / BATCH_SIZE))).prefetch(1)

train_ds = make_dataset(x_train, y_train, shuffle=True)
val_ds = make_dataset(x_val, y_val, shuffle=False)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
model = tf.keras.Sequential([
    tf.keras.layers.Input((N_FRAMES, 126)),
    tf.keras.layers.Conv1D(96, 5, padding='same', activation='relu'),
    tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(N_CLASSES, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()
print(f'Uniform random accuracy reference: {1 / N_CLASSES:.2%}')


## Train

Executing the next cell starts training. The best checkpoint is selected by validation accuracy. Early stopping limits epochs without improvement. Enable the full-dataset configuration only after the small pipeline check succeeds.


In [ ]:
checkpoint_path = RUN_DIR / 'best.keras'
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            str(checkpoint_path), monitor='val_accuracy', mode='max', save_best_only=True),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(str(RUN_DIR / 'training_log.csv')),
    ])
(RUN_DIR / 'history.json').write_text(
    json.dumps({k: [float(v) for v in values] for k, values in history.history.items()}, indent=2),
    encoding='utf-8')
print('Saved best model:', checkpoint_path)


In [ ]:
best_model = tf.keras.models.load_model(checkpoint_path)
scores = {k: float(v) for k, v in best_model.evaluate(val_ds, return_dict=True).items()}
# Keep prediction order identical to validation_split.csv.
prediction_ds = val_ds.map(lambda features, targets: features)
probabilities = best_model.predict(prediction_ds)
predictions = probabilities.argmax(axis=1)
assert len(predictions) == len(y_val)
metrics = dict(scores, pipeline_check=SAMPLE_LIMIT is not None,
               validation_samples=len(y_val),
               best_epoch=int(np.argmax(history.history['val_accuracy']) + 1))
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
inverse_labels = {index: sign for sign, index in labels.items()}
class_names = [inverse_labels[i] for i in range(N_CLASSES)]
report = classification_report(
    y_val, predictions, labels=list(range(N_CLASSES)),
    target_names=class_names, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(RUN_DIR / 'classification_report.csv')
results = val_rows[['path', 'participant_id', 'sign']].copy()
results['predicted_sign'] = [inverse_labels[int(i)] for i in predictions]
results['confidence'] = probabilities.max(axis=1)
results.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)
print(json.dumps(metrics, indent=2))
if SAMPLE_LIMIT is not None:
    print('PIPELINE CHECK ONLY: rerun with SAMPLE_LIMIT=None for a full-task baseline.')
display(results.head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, title, axis in [('loss', 'Loss', axes[0]), ('accuracy', 'Accuracy', axes[1])]:
    epoch_numbers = range(1, len(history.history[key]) + 1)
    axis.plot(epoch_numbers, history.history[key], label='train')
    axis.plot(epoch_numbers, history.history['val_' + key], label='validation')
    axis.set(title=title, xlabel='Epoch', ylabel=title)
    axis.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'learning_curves.png', dpi=150)
plt.show()

matrix = confusion_matrix(y_val, predictions, labels=list(range(N_CLASSES)))
np.save(RUN_DIR / 'confusion_matrix.npy', matrix)
np.fill_diagonal(matrix, 0)
confusions = [(int(matrix[i, j]), class_names[i], class_names[j])
              for i, j in zip(*np.nonzero(matrix))]
confusions.sort(reverse=True)
display(pd.DataFrame(confusions[:15], columns=['count', 'true_sign', 'predicted_sign']))
print('Artifacts saved to:', RUN_DIR)


## After the pipeline check

1. Set `SAMPLE_LIMIT = None` and `EPOCHS = 20`, then rerun from the configuration cell. A new output directory is created for each run.
2. Record the full-data validation accuracy, signer split, best epoch, and training curves. Compare future changes using the same split.
3. Download the best model, label map, run configuration, metrics, and reports before ending the Kaggle session. Keep large feature caches and model files out of Git.
4. Add measured results to the README only after inspecting the completed run.

This checkpoint accepts preprocessed `(32, 126)` features. It is not a competition-ready TFLite submission and does not include camera input or sentence translation.
